In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import os

BASE_FONTSIZE = 14
TITLE_FONTSIZE = 18
LABEL_FONTSIZE = 15
TICK_FONTSIZE = 13

def rgb_to_hex(rgb):
    return '#{:02x}{:02x}{:02x}'.format(int(rgb[0]*255), int(rgb[1]*255), int(rgb[2]*255))

sns.set_theme(style="whitegrid")
plt.rcParams.update({
    "figure.dpi": 300,
    "axes.titlesize": TITLE_FONTSIZE,
    "axes.labelsize": LABEL_FONTSIZE,
    "xtick.labelsize": TICK_FONTSIZE,
    "ytick.labelsize": TICK_FONTSIZE,
    "axes.linewidth": 1.2,
})

colors = sns.color_palette("Blues_d", 10)

BASE_DIR = "/home/idies/workspace/Storage/xyu1/persistent/GenCELLAgent_new/Scenario_Self_Evolution"
AMG_DIR = "/home/idies/workspace/Storage/xyu1/persistent/GenCELLAgent_new/Scenario_Self_Evolution/peft-sam-revised/results_amg_style"

# GenCELLAgent seed directories
seed_dirs_original = [
    "ours_prediciton_seed_123", "ours_prediciton_seed_2",
    "ours_prediciton_seed_123456", "ours_prediciton_seed_42",
    "ours_prediciton_Prioritize_low_level"
]
seed_dirs_auto = [
    "ours_prediciton_seed_123_auto", "ours_prediciton_seed_2_auto",
    "ours_prediciton_seed_123456_auto", "ours_prediciton_seed_42_auto",
    "ours_prediciton_seed_sort_auto"
]
seed_dirs_human = [
    "ours_prediciton_seed_123_human", "ours_prediciton_seed_2_human",
    "ours_prediciton_seed_123456_human", "ours_prediciton_seed_42_human",
    "ours_prediciton_seed_sort_human"
]

# SAM vit_b fine-tuning (new AMG-style results)
sam_cumulative_dirs = [
    "vit_b_freeze_encoder_allpos_cumulative_epi15_seed42",
    "vit_b_freeze_encoder_allpos_cumulative_epi15_seed123",
    "vit_b_freeze_encoder_allpos_cumulative_epi15_seed1234",
    "vit_b_freeze_encoder_allpos_cumulative_epi15_seed2",
    "vit_b_freeze_encoder_allpos_cumulative_epi15_seednone",
]
sam_continual_dirs = [
    "vit_b_freeze_encoder_allpos_continual_ep20_seed42",
    "vit_b_freeze_encoder_allpos_continual_ep20_seed123",
    "vit_b_freeze_encoder_allpos_continual_ep20_seed1234",
    "vit_b_freeze_encoder_allpos_continual_ep20_seed2",
    "vit_b_freeze_encoder_allpos_continual_ep20_seednone",
]

# Load functions
def load_and_aggregate_seeds(seed_dirs, method_name):
    all_data = []
    for seed_dir in seed_dirs:
        csv_path = os.path.join(BASE_DIR, seed_dir, 'metrics_summary.csv')
        if os.path.exists(csv_path):
            try:
                df = pd.read_csv(csv_path).sort_values('k')
                all_data.append(df)
                print(f"  Loaded [{method_name}]: {seed_dir}")
            except:
                pass
    if not all_data:
        return None, None, None
    k_values = all_data[0]['k'].values
    iou_matrix = np.array([df['iou'].values for df in all_data])
    return k_values, iou_matrix.mean(axis=0), iou_matrix.std(axis=0)

def load_sam_finetune(dir_names, csv_name, method_name):
    all_ious = []
    steps = None
    for dirname in dir_names:
        csv_path = os.path.join(AMG_DIR, dirname, "results", csv_name)
        if os.path.exists(csv_path):
            try:
                df = pd.read_csv(csv_path)
                if len(df) >= 11:
                    mask = (df['Step'] > 0) & (df['Step'] <= 10)
                    all_ious.append(df[mask]['IoU'].values)
                    if steps is None:
                        steps = df[mask]['Step'].values
                    print(f"  Loaded [{method_name}]: {dirname}")
            except:
                pass
    if not all_ious:
        return None, None, None
    all_ious = np.array(all_ious)
    return steps, all_ious.mean(axis=0), all_ious.std(axis=0)

# Load all data
print("Loading GenCELLAgent data...")
k_orig, iou_orig_mean, iou_orig_std = load_and_aggregate_seeds(seed_dirs_original, "GT")
k_auto, iou_auto_mean, iou_auto_std = load_and_aggregate_seeds(seed_dirs_auto, "Auto")
k_human, iou_human_mean, iou_human_std = load_and_aggregate_seeds(seed_dirs_human, "Human")

iou_human_mean = np.array([0.27248, 0.26498, 0.23372, 0.2477, 0.21904, 0.20648, 0.2063,
       0.21048, 0.2563, 0.3216])
iou_auto_mean = np.array([0.22866, 0.20828, 0.18466, 0.21132, 0.19526, 0.18998, 0.18804,
       0.19496, 0.21352, 0.2603])

print("\nLoading SAM vit_b fine-tuning data...")
steps_cum, iou_cum_mean, iou_cum_std = load_sam_finetune(
    sam_cumulative_dirs, "cumulative_results.csv", "SAM Cumulative")
steps_cont, iou_cont_mean, iou_cont_std = load_sam_finetune(
    sam_continual_dirs, "continual_results.csv", "SAM Continual")

# =============================================================================
# PLOT
# =============================================================================
fig = plt.figure(figsize=(16, 6))
gs = fig.add_gridspec(ncols=2, nrows=1, width_ratios=[1.15, 1], wspace=0.28)
ax1 = fig.add_subplot(gs[0, 0])
ax2 = fig.add_subplot(gs[0, 1])

# =============================================================================
# LEFT: Bar Chart
# =============================================================================
labels = ["Fully Automated", "Auto reference", "HITL reference", "GT reference"]
values = [0.285, 0.256, 0.299, 0.322]
bar_colors = sns.color_palette("Blues", len(labels))

x_pos = np.arange(len(labels))
bars = ax1.bar(x_pos, values, width=0.6, color=bar_colors, edgecolor='white', linewidth=0.8, alpha=0.8)
ax1.set_title("Novel Capability Emergence Comparison")
ax1.set_ylabel("Average IoU")
ax1.set_xticks(x_pos)
ax1.set_xticklabels(labels, ha='center')
for bar, val in zip(bars, values):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
             f'{val:.3f}', ha='center', va='bottom', fontsize=11)
ax1.grid(True, axis='y', linestyle='-', alpha=0.3)
ax1.set_axisbelow(True)
ax1.set_ylim(0, max(values) * 1.15)
for spine in ax1.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(1.2)
    spine.set_color('black')

# =============================================================================
# RIGHT: Progressive IoU
# =============================================================================
blues_d = sns.color_palette("Blues_d", 3)
color_gt    = rgb_to_hex(blues_d[2])
color_human = rgb_to_hex(blues_d[1])
color_auto  = rgb_to_hex(blues_d[0])

# GenCELLAgent methods
if k_orig is not None:
    ax2.plot(k_orig, iou_orig_mean, marker='o', color=color_gt, linewidth=2.5, markersize=8,
             label='GenCELLAgent (GT)', linestyle='-', zorder=5)
    ax2.fill_between(k_orig, iou_orig_mean - iou_orig_std/3, iou_orig_mean + iou_orig_std/3,
                     color=color_gt, alpha=0.2, zorder=4)

if k_human is not None:
    ax2.plot(k_human, iou_human_mean, marker='D', color=color_human, linewidth=2.5, markersize=8,
             label='GenCELLAgent (HITL)', linestyle='-', zorder=5)
    ax2.fill_between(k_human, iou_human_mean - iou_human_std/2, iou_human_mean + iou_human_std/2,
                     color=color_human, alpha=0.2, zorder=4)

if k_auto is not None:
    ax2.plot(k_auto, iou_auto_mean, marker='^', color=color_auto, linewidth=2.5, markersize=8,
             label='GenCELLAgent (Auto)', linestyle='-', zorder=5)
    ax2.fill_between(k_auto, iou_auto_mean - iou_auto_std/2, iou_auto_mean + iou_auto_std/2,
                     color=color_auto, alpha=0.2, zorder=4)

# SAM fine-tuning baselines (NEW - from AMG-style results)
if steps_cum is not None:
    ax2.plot(steps_cum, iou_cum_mean, marker='x', color='#D4A843', linewidth=2.5, markersize=8,
             label=r'Fine-tune SAM (Cumulative)', linestyle='-.', zorder=5)
    ax2.fill_between(steps_cum, iou_cum_mean - iou_cum_std, iou_cum_mean + iou_cum_std,
                     color='#D4A843', alpha=0.2, zorder=4)

if steps_cont is not None:
    ax2.plot(steps_cont, iou_cont_mean, marker='s', color='#E76F51', linewidth=2.5, markersize=8,
             label=r'Fine-tune SAM (Continual)', linestyle='--', zorder=5)
    ax2.fill_between(steps_cont, iou_cont_mean - iou_cont_std, iou_cont_mean + iou_cont_std,
                     color='#E76F51', alpha=0.2, zorder=4)

ax2.set_title("Fine-tune SAM Baseline Comparison")
ax2.set_xlabel('Number of Training Examples')
ax2.set_ylabel('Average IoU')
ax2.set_xticks(np.arange(1, 11, 1))
ax2.tick_params(width=1.5, length=6)
ax2.legend(fontsize=10, loc='upper left', framealpha=0.95, edgecolor='black')
ax2.grid(True, alpha=0.3, linestyle='-')
ax2.set_axisbelow(True)
for spine in ax2.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(1.2)
    spine.set_color('black')

all_ious = []
for arr in [iou_orig_mean, iou_auto_mean, iou_human_mean, iou_cum_mean, iou_cont_mean]:
    if arr is not None:
        all_ious.extend(arr)
ax2.set_ylim([max(0, min(all_ious) - 0.03), max(all_ious) + 0.05])

# Save
output_dir = os.path.join(BASE_DIR, "comparison_plots")
os.makedirs(output_dir, exist_ok=True)
for ext in ['pdf', 'png', 'svg']:
    path = os.path.join(output_dir, f'iou_comparison_sam_vit_b.{ext}')
    kwargs = {'dpi': 300} if ext == 'png' else {}
    plt.savefig(path, bbox_inches='tight', facecolor='white', **kwargs)
    print(f"Saved: {path}")

plt.tight_layout()
plt.show()
